# Merging a LoRA Adapter into the Base Model (Decoder / Causal LM)

We will work on the decoder (causal language model) variant of the LoRA merge. Compared to the encoder merge, the decoder merge is simpler because there is no classification head to size. The base model is an `AutoModelForCausalLM` that generates text, so the merge is a straightforward weight fusion.

## Encoder vs Decoder Merge

| | Encoder merge | Decoder merge (this notebook) |
|---|---|---|
| Model class | `AutoModelForSequenceClassification` | `AutoModelForCausalLM` |
| Label mapping | Must reconstruct `num_labels`, `id2label`, `label2id` from training data | Not needed |
| Tokenizer source | Loaded from base model | Loaded from adapter path (may contain added tokens) |
| dtype | `float32`| `float16` (matches training precision) |
| Serialization | Default | `safe_serialization=True` (safetensors format) |

## What you will learn
- Why the tokenizer is loaded from the adapter path instead of the base model
- How to load a causal LM in float16 for merging
- How `merge_and_unload` fuses LoRA weights into the base model
- How to save the merged model in safetensors format for serving

---
## 1 Imports

| Library | Purpose |
|---|---|
| `transformers` | Load the base causal language model (`AutoModelForCausalLM`) and tokenizer |
| `peft` | Load the LoRA adapter and perform the merge (`PeftModel`) |
| `torch` | Specify `float16` dtype to match training precision |

In [1]:
!pip install transformers==4.57.1 peft==0.17.0 trl==0.21.0 accelerate==1.10.0


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import torch
import os
import gc
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

/opt/app-root/lib64/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


This notebook loads the base model onto GPU so the merge math runs entirely in VRAM. This keeps system RAM free for the Jupyter server and avoids OOM kills or 504 timeouts during long running cells. The workbench needs at least one CUDA device with enough VRAM for the model in float16 (roughly 6 GB for a 3B model).

In [3]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "no CUDA device found. this notebook requires a GPU workbench. "
        "use the encoder merge notebook for CPU only environments."
    )

total_vram = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {total_vram:.1f} GB")
print(f"CUDA version: {torch.version.cuda}")

GPU: Tesla T4
VRAM: 14.6 GB
CUDA version: 13.0


---
## 2 Configuration

Three paths are needed:
- `base_model`: the Hugging Face model ID or local path for the pre trained causal LM (e.g. Qwen 2.5 3B Instruct)
- `adapter_path`: where the LoRA adapter checkpoint was saved after training
- `output_path`: where the merged model will be written

In [4]:
# update these paths for your environment
base_model_id = os.environ.get("BASE_MODEL", "Qwen/Qwen2.5-3B-Instruct")
adapter_path = os.environ.get("ADAPTER_PATH", "/mnt/data/adapters/latest")
output_path = os.environ.get("OUTPUT_PATH", "/mnt/data/merged_model")

print(f"base model: {base_model_id}")
print(f"adapter:    {adapter_path}")
print(f"output:     {output_path}")

base model: Qwen/Qwen2.5-3B-Instruct
adapter:    ./mnt/data/adapters/latest
output:     ./mnt/data/merged_model


---
## 3 Loading the Tokenizer from the Adapter Path

This is the key difference from the encoder merge. During causal LM fine tuning, the tokenizer is sometimes modified. For example:
- Special tokens may be added for chat templates or structured output markers
- The padding token may be set or changed
- Custom chat template formatting may be saved alongside the adapter

Loading the tokenizer from the adapter path ensures that the merged model gets the exact tokenizer that was used during training, including any of these modifications. If we loaded from the base model instead, these customizations would be lost and inference would produce different results.

In [5]:
print(f"loading customized tokenizer from: {adapter_path}")
tokenizer = AutoTokenizer.from_pretrained(adapter_path, trust_remote_code=True)

print(f"vocab size: {len(tokenizer)}")
print(f"pad token: {tokenizer.pad_token}")
print(f"eos token: {tokenizer.eos_token}")

loading customized tokenizer from: ./mnt/data/adapters/latest
vocab size: 151665
pad token: <|im_end|>
eos token: <|im_end|>


---
## 4 Loading the Base Model

The base model is loaded in `float16` to match the precision used during training. This matters because the LoRA deltas were computed relative to float16 base weights, so merging at a different precision could introduce small numerical differences.

Unlike the encoder variant, there is no classification head to configure. The causal LM head (which maps hidden states to vocabulary logits) is part of the pre trained model and was not modified by LoRA training.

Key parameters:
- `dtype=torch.float16`: matches training precision
- `low_cpu_mem_usage=True`: uses the accelerate lazy loading path to reduce peak RAM during loading. Weights are materialized on demand instead of all at once
- `trust_remote_code=True`: needed for models like Qwen that ship custom code

In [6]:
print(f"loading base model: {base_model_id}")
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

print(f"model dtype: {next(base_model.parameters()).dtype}")
print(f"model device: {next(base_model.parameters()).device}")
print(f"total params: {sum(p.numel() for p in base_model.parameters()):,}")
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

loading base model: Qwen/Qwen2.5-3B-Instruct


`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [00:46<00:00, 23.04s/it]

model dtype: torch.float16
model device: cuda:0
total params: 3,085,938,688
GPU memory used: 5.7 GB


---
## 5 Loading the LoRA Adapter

`PeftModel.from_pretrained` attaches the adapter's low rank matrices (A and B) to the target modules in the base model. At this point the model still has two separate sets of weights and PEFT computes `W_base + (alpha/r) * B @ A` on the fly during each forward pass.

In [7]:
print(f"loading adapter: {adapter_path}")
model = PeftModel.from_pretrained(base_model, adapter_path)

# show adapter details
peft_config = model.peft_config["default"]
print(f"adapter rank (r): {peft_config.r}")
print(f"adapter alpha: {peft_config.lora_alpha}")
print(f"target modules: {peft_config.target_modules}")
print(f"trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

loading adapter: ./mnt/data/adapters/latest
adapter rank (r): 16
adapter alpha: 32
target modules: {'o_proj', 'v_proj', 'gate_proj', 'up_proj', 'down_proj', 'k_proj', 'q_proj'}
trainable params: 0


---
## 6 Merge and Unload

This is the same core operation as in the encoder merge. For each target module, `merge_and_unload()` computes:

```
W_merged = W_base + (alpha / r) * B @ A
```

and replaces the original linear layer with a standard `nn.Linear` containing the merged weight. The PEFT wrapper is then removed, leaving a plain `transformers` model.

After this call:
- The model no longer needs the PEFT library to run
- Forward passes are slightly faster (no per step adapter computation)
- The model can be served with vLLM, TGI, KServe or any framework that loads standard Hugging Face checkpoints

In [8]:
print("merging weights...")
model = model.merge_and_unload()

# free the original base model ref and reclaim GPU memory
del base_model
gc.collect()
torch.cuda.empty_cache()

print(f"model type after merge: {type(model).__name__}")
print(f"total params: {sum(p.numel() for p in model.parameters()):,}")

merging weights...
model type after merge: Qwen2ForCausalLM
total params: 3,085,938,688


---
## 7 Saving the Merged Model

The merged model is saved with `safe_serialization=True` which writes weights in the safetensors format rather than pickle based `pytorch_model.bin`. Safetensors is the recommended format because:
- It prevents arbitrary code execution on load (unlike pickle)
- It supports zero copy memory mapping for faster loading
- It is the default expected by serving frameworks like vLLM

`max_shard_size="2GB"` splits the weights into multiple shard files instead of materializing one large tensor blob in memory. This keeps peak RAM during serialization well below the full model size and avoids OOM kills on memory constrained notebook environments.

The tokenizer is saved alongside the model so the output directory is a complete self contained checkpoint.

In [9]:
print(f"saving merged model to: {output_path}")
model.save_pretrained(output_path, safe_serialization=True, max_shard_size="2GB")
tokenizer.save_pretrained(output_path)

# list the saved files
saved_files = os.listdir(output_path)
print(f"\nsaved {len(saved_files)} files:")
for fname in sorted(saved_files):
    fpath = os.path.join(output_path, fname)
    size_mb = os.path.getsize(fpath) / (1024 * 1024)
    print(f"  {fname:40s} {size_mb:8.1f} MB")

print("\ndone. the model is ready for standard Hugging Face inference.")

saving merged model to: ./mnt/data/merged_model

saved 14 files:
  added_tokens.json                             0.0 MB
  chat_template.jinja                           0.0 MB
  config.json                                   0.0 MB
  generation_config.json                        0.0 MB
  merges.txt                                    1.6 MB
  model-00001-of-00004.safetensors           1873.6 MB
  model-00002-of-00004.safetensors           1868.2 MB
  model-00003-of-00004.safetensors           1868.2 MB
  model-00004-of-00004.safetensors            276.0 MB
  model.safetensors.index.json                  0.0 MB
  special_tokens_map.json                       0.0 MB
  tokenizer.json                               10.9 MB
  tokenizer_config.json                         0.0 MB
  vocab.json                                    2.6 MB

done. the model is ready for standard Hugging Face inference.


---
## 8 Quick Verification (Optional)

Reload the merged model from disk and run a quick generation to confirm everything works without PEFT.

In [10]:
# free the merged model before reloading to avoid holding two copies
del model
del tokenizer
gc.collect()
torch.cuda.empty_cache()

print("reloading merged model from disk for verification...")
reloaded_model = AutoModelForCausalLM.from_pretrained(
    output_path,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,    
)
reloaded_tokenizer = AutoTokenizer.from_pretrained(output_path, trust_remote_code=True)

# run a sample generation
messages = [
    {"role": "system", "content": "You are an AI intent classifier. Output only valid JSON."},
    {"role": "user", "content": "ขอเปิดใช้งานตอนนี้เลยค่ะ จะได้เริ่มใช้เบอร์ได้ แล้วช่วยดูโปรใหม่มีอะไรบ้างค่ะ ของเก่ารู้สึกโปรมันแพงไป T T"}
]

prompt = reloaded_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = reloaded_tokenizer(prompt, return_tensors="pt").to(reloaded_model.device)

with torch.no_grad():
    outputs = reloaded_model.generate(
        **inputs,
        max_new_tokens=50,
        temperature=0.0,
        do_sample=False,
    )

# decode only the generated tokens
prompt_length = inputs["input_ids"].shape[1]
generated = reloaded_tokenizer.decode(outputs[0][prompt_length:], skip_special_tokens=True)

print(f"input:  ขอเปิดใช้งานตอนนี้เลยค่ะ จะได้เริ่มใช้เบอร์ได้ แล้วช่วยดูโปรใหม่มีอะไรบ้างค่ะ ของเก่ารู้สึกโปรมันแพงไป T T")
print(f"output: {generated}")

reloading merged model from disk for verification...


Loading checkpoint shards: 100%|██████████| 4/4 [00:53<00:00, 13.35s/it]
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


input:  ขอเปิดใช้งานตอนนี้เลยค่ะ จะได้เริ่มใช้เบอร์ได้ แล้วช่วยดูโปรใหม่มีอะไรบ้างค่ะ ของเก่ารู้สึกโปรมันแพงไป T T
output: {"intent": "MOBILE_SIM_ACTIVATION_POSTPAID"}


---
## 9 Upload Merged Model to S3

Once the merged model is saved locally, we upload the entire output directory to an S3 bucket so it can be pulled by serving infrastructure or shared across environments.

Configuration is controlled by three environment variables:
- `S3_BUCKET`: the target S3 bucket name
- `S3_PREFIX`: the key prefix (folder path) inside the bucket
- `AWS_DEFAULT_REGION`: region for the S3 client (defaults to `us-east-1`)

Standard AWS credentials (`AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY` or an instance profile) must be available in the environment.

In [11]:
!pip install boto3


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [13]:
import boto3
from botocore.config import Config

os.environ["AWS_ACCESS_KEY_ID"] = os.environ.get("AWS_ACCESS_KEY_ID", "") # your access key
os.environ["AWS_SECRET_ACCESS_KEY"] = os.environ.get("AWS_SECRET_ACCESS_KEY", "") # your secret
os.environ["AWS_ENDPOINT_URL"] = os.environ.get("AWS_ENDPOINT_URL", "") # url of the bucket --> http://s3.openshift-storage.svc.cluster.local

s3_bucket = os.environ.get("S3_BUCKET", "qwen25-3b-model-bucket")
s3_prefix = os.environ.get("S3_PREFIX", "merged-models/decoder")
region = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")

s3 = boto3.client("s3", config=Config(region_name=region))

# walk the output directory and upload every file
uploaded = []
for root, _dirs, files in os.walk(output_path):
    for fname in sorted(files):
        local_path = os.path.join(root, fname)
        relative = os.path.relpath(local_path, output_path)
        s3_key = f"{s3_prefix}/{relative}"

        size_mb = os.path.getsize(local_path) / (1024 * 1024)
        print(f"uploading {relative} ({size_mb:.1f} MB) -> s3://{s3_bucket}/{s3_key}")
        s3.upload_file(local_path, s3_bucket, s3_key)
        uploaded.append(s3_key)

print(f"\nuploaded {len(uploaded)} files to s3://{s3_bucket}/{s3_prefix}/")
print("done.")

uploading added_tokens.json (0.0 MB) -> s3://qwen25-3b-model-bucket/merged-models/decoder/added_tokens.json
uploading chat_template.jinja (0.0 MB) -> s3://qwen25-3b-model-bucket/merged-models/decoder/chat_template.jinja
uploading config.json (0.0 MB) -> s3://qwen25-3b-model-bucket/merged-models/decoder/config.json
uploading generation_config.json (0.0 MB) -> s3://qwen25-3b-model-bucket/merged-models/decoder/generation_config.json
uploading merges.txt (1.6 MB) -> s3://qwen25-3b-model-bucket/merged-models/decoder/merges.txt
uploading model-00001-of-00004.safetensors (1873.6 MB) -> s3://qwen25-3b-model-bucket/merged-models/decoder/model-00001-of-00004.safetensors
uploading model-00002-of-00004.safetensors (1868.2 MB) -> s3://qwen25-3b-model-bucket/merged-models/decoder/model-00002-of-00004.safetensors
uploading model-00003-of-00004.safetensors (1868.2 MB) -> s3://qwen25-3b-model-bucket/merged-models/decoder/model-00003-of-00004.safetensors
uploading model-00004-of-00004.safetensors (276.0

---
## Summary

| Step | What it does |
|---|---|
| Tokenizer load | Loads from adapter path to preserve any custom tokens or chat template changes |
| Base model load | Loads causal LM in float16 to match training precision |
| Adapter load | Attaches the LoRA low rank matrices to the base model |
| Merge and unload | Fuses adapter deltas into the base weights via `W + (alpha/r) * B @ A` |
| Save | Writes a standalone checkpoint in safetensors format |
| Verify | Reloads from disk and runs a generation to confirm the merge is correct |
| Upload to S3 | Pushes the full merged checkpoint to an S3 bucket via boto3 |

The merged decoder model is now ready for deployment.